---
title: "Gas–solid fluidized bed: online CFD–DEM coupling"
subtitle: "Blow gas up through a cylinder of grains until the bed unpacks and boils — the two solvers exchanging drag and void fraction every step — then check the granular solver still flies at a million grains."
author: "Peclet"
date: "2026-07-05"
categories: [coupling, cfd-dem, dem, flow, fluidization, gidaspow, sdf, performance]
jupyter: python3
---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/computational-chemical-engineering/peclet-examples/blob/main/examples/fluidized-bed/index.ipynb){target="_blank"}
&nbsp;The two single-phase examples ([packed bed](../random-packed-bed/index.qmd), [drum](../rotating-drum/index.qmd)) are *offline* (dem → flow). This one is **online**: the CFD and DEM solvers run together and exchange forces every step, so it needs the `peclet-coupling` module (built from source — see *Reproduce*).

## What you'll learn

Everything so far coupled the two solvers **offline**: pack grains with `peclet.dem`, freeze them, then
solve the flow. A **fluidized bed** cannot be split that way — the grains move *because* of the gas and
the gas slows *because* of the grains. `peclet.coupling.CfdDem` runs the **unresolved point-particle
CFD–DEM** loop: every fluid step it deposits the grains' volume onto the grid (→ void fraction ε),
evaluates a drag law, pushes the reaction onto the fluid and the drag onto the grains, sub-steps the
DEM, and advances the flow. You will:

1. Build a **cylindrical vessel** whose signed distance field is used **twice** — a cut-cell IBM
   no-slip wall for the gas *and* a restitution+friction wall for the grains.
2. Drive it with a **gas inflow at the bottom** and an **outflow at the top**, and contain the grains
   with a **bouncy distributor** and a **top lid the gas passes straight through**.
3. Close the momentum exchange with the **Gidaspow** drag correlation (Ergun in the dense packing,
   Wen & Yu in the dilute freeboard) on a **volume-weighted** void fraction, and watch the bed
   **fluidize** — expand and bubble — once the gas velocity clears minimum fluidization.
4. Ask the blunt performance question: **is the granular solver actually fast?** We benchmark the
   ArborX bounding-volume hierarchy up to **a million grains**.

In [ ]:
#| label: bootstrap
#| code-summary: "Environment bootstrap (local build, or peclet + peclet-coupling)"
import importlib.util, os, subprocess, sys
_local = os.environ.get("PECLET_LOCAL_BUILD")
if _local:
    for p in _local.split(os.pathsep):
        sys.path.insert(0, p)
elif importlib.util.find_spec("peclet") is None:
    # peclet-coupling is sdist-only (it builds its Kokkos kernels from source against a Kokkos prefix);
    # on a plain runtime install the CPU family + the coupling extra.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peclet[cfd-dem]"], check=True)

In [ ]:
import numpy as np
import time
import matplotlib.pyplot as plt
from peclet import flow, dem
from peclet.dem import build_wall_sdf
from peclet.coupling import CfdDem

plt.rcParams.update({"figure.dpi": 130, "font.size": 10, "axes.axisbelow": True,
                     "figure.facecolor": "white", "savefig.bbox": "tight"})
print("backends — flow:", flow.execution_space, " dem:", dem.execution_space)

## The setup — one cylinder, two solvers

We work in **grid-cell units** (cell size `h = 1`). The vessel is a cylinder of radius `R` along the
`z` axis, sitting in an `NX × NY × NZ` box. The grains are **spheres** of diameter `dp = 0.2` cells —
the *unresolved* CFD–DEM requirement is a cell about **five particle diameters** wide, so the void
fraction each cell reports is a meaningful average over many grains.

In [ ]:
NX = NY      = 8               # lateral grid
NZ           = 18              # tall column (bed + freeboard)
R, H_wall    = 2.5, 12.0       # vessel radius; grain-containment lid height (< NZ -> gas freeboard)
dp           = 0.2             # grain diameter -> cell / dp = 5
rp           = dp / 2
cx, cy       = NX / 2.0, NX / 2.0
rho_g, mu_g  = 1.0, 0.05       # gas
rho_p, grav  = 40.0, 2.0e-3    # grains (rho_p >> rho_g, like sand in air) + gravity
m_p          = rho_p * (4/3) * np.pi * rp**3

The **same radial SDF** — positive in the fluid, negative in the wall — is what both solvers collide
against. For the gas it becomes a cut-cell immersed no-slip wall; for the grains it becomes an SDF
wall with its own restitution and friction.

In [ ]:
def cylinder_flow_sdf():
    "Grid SDF (x-fastest): >0 in the fluid inside the vessel, <0 in the wall outside R."
    X, Y, _ = np.meshgrid(np.arange(NX) + .5, np.arange(NY) + .5, np.arange(NZ) + .5, indexing="ij")
    return (R - np.hypot(X - cx, Y - cy)).astype(np.float64)

def capped_cylinder_wall(pts):
    "Particle wall f(points)->distance, >0 in the void: inside R, above the distributor, below the lid."
    return np.minimum.reduce([R - np.hypot(pts[:, 0] - cx, pts[:, 1] - cy),
                              pts[:, 2] - 0.0, H_wall - pts[:, 2]])

**The gas** gets the cylinder as an immersed body plus an inflow floor and an outflow roof. Face codes
are `-x,+x,-y,+y,-z,+z`; the `-z` floor is an inflow at the superficial velocity `U`, the `+z` roof an
outflow, and the lateral faces no-slip (the fluid never reaches them — the cylinder confines it).

In [ ]:
def make_flow(U):
    s = flow.Solver(NX, NY, NZ)
    s.set_rho(rho_g); s.set_mu(mu_g); s.set_dt(0.05)
    s.set_domain_bc(4, 2, 0.0, 0.0, U)     # -z: inflow, gas up at U
    s.set_domain_bc(5, 3)                  # +z: outflow
    for f in (0, 1, 2, 3): s.set_domain_bc(f, 1)   # lateral no-slip
    s.set_solid(cylinder_flow_sdf().flatten(order="F"), True)   # cut-cell IBM no-slip vessel
    return s

::: {.callout-tip}
## Any length unit works — the halo follows the grain
You just set `radius = rp`. The DEM sizes its contact-search radius and periodic/rank **ghost band**
from the *actual* grain radius (`scale × global_scale × baseRadius`), so a grain 0.2 cells across — or
`1e-3 m` in SI, with `global_scale` left at 1 — gets a halo band proportional to it, with no scaling
gymnastics. (Earlier releases tied the band to `global_scale` alone; a grain much smaller than it made
the broadphase search a huge radius.)
:::

**The grains** get gravity, a particle–particle material, and the capped-cylinder wall — a **bouncy
distributor** at the bottom (restitution 0.7, ≠ 1) and a **lid** that stops grains but is invisible to
the gas (which leaves through the outflow roof above it).

In [ ]:
def make_dem(pos):
    n = len(pos)
    d = dem.Simulation(int(2.2 * n) + 256)
    d.initialize(shape_type=1, radius=rp)        # sphere at the real grain radius (any unit works)
    d.set_domain((0, 0, 0), (NX, NY, NZ)); d.enable_periodicity(False, False, False)
    d.set_gravity(0, 0, -grav)
    d.set_material_params(0.8, 0.0, 0.2)          # particle-particle restitution / friction
    d.set_dt(0.05 / 20)
    build_wall_sdf(capped_cylinder_wall, ((0, 0, 0), (NX, NY, NZ)), resolution=64) \
        .add_to(d, restitution=0.7, friction=0.3)  # bouncy distributor + lid + side wall
    d.set_positions(np.c_[pos, np.full(n, 1.0 / m_p, np.float32)])
    d.set_velocities(np.zeros((n, 3), np.float32))
    return d

def initial_packing(n_bed=3.0, solid_frac=0.45):
    "Loose grains inside the vessel, z in (rp, n_bed]; they settle into a packed bed."
    vp = (4/3) * np.pi * rp**3
    npart = int(solid_frac * np.pi * R**2 * n_bed / vp)
    rng, out, n = np.random.default_rng(7), np.empty((npart, 3), np.float32), 0
    while n < npart:
        c = rng.uniform([cx - R, cy - R, rp], [cx + R, cy + R, n_bed], size=(npart, 3))
        c = c[(c[:, 0] - cx)**2 + (c[:, 1] - cy)**2 < (R - rp)**2]
        out[n:n + len(c)] = c[:npart - n]; n += len(c)
    return out

The coupling itself is one object. `drag="gidaspow"` selects the Ergun/Wen & Yu blend; the void
fraction is deposited by **volume weighting** (trilinear solid-volume scatter). `move_particles=True`
turns on the two-way loop.

In [ ]:
pos0  = initial_packing()
print(f"{len(pos0)} grains, dp={dp} (cell/dp={1/dp:.0f}), R={R}")

def bed_top(cpl):
    "95th-percentile grain height — the top of the bed."
    z = np.asarray(cpl._particles()[0])[:, 2]
    return float(np.percentile(z, 95)) if z.size else 0.0

## Fluidize it

Below minimum fluidization the gas trickles through a fixed bed; above it, the drag carries the grain
weight and the bed **unpacks and expands**. We drive it well above `U_mf` and record the bed height and
a couple of side-view snapshots.

In [ ]:
#| label: fig-fluidize
#| fig-cap: "The bed fluidizes: a settled packing (left) expands and boils as gas drives up through it (right); grains coloured by speed. The 95th-percentile bed height rises ~50%."
s   = make_flow(0.25)
d   = make_dem(pos0)
cpl = CfdDem(s, d, fluid_dt=0.05, mu=mu_g, rho=rho_g, radius=rp, drag="gidaspow",
             dem_substeps=20, eps_min=0.3, periodic=(False, False, False), move_particles=True)

h0, hist, snaps = bed_top(cpl), [], {}
for i in range(120):
    cpl.step()
    hist.append(bed_top(cpl))
    if i in (0, 119):
        p = np.asarray(cpl._particles()[0]); v = np.asarray(cpl._particles()[1])
        snaps[i] = (p.copy(), np.linalg.norm(v, axis=1))

fig, ax = plt.subplots(1, 3, figsize=(9, 3.4), gridspec_kw={"width_ratios": [1, 1, 1.3]})
for k, (i, title) in enumerate([(0, "settled bed"), (119, "fluidized")]):
    p, spd = snaps[i]
    sc = ax[k].scatter(p[:, 0], p[:, 2], c=spd, s=3, cmap="viridis", vmin=0, vmax=np.percentile(spd, 98))
    ax[k].add_patch(plt.Rectangle((cx - R, 0), 2 * R, H_wall, fill=False, ec="0.6", lw=1))
    ax[k].set(title=title, xlim=(0, NX), ylim=(0, NZ), xlabel="x", aspect="equal")
    ax[k].set_ylabel("z" if k == 0 else "")
ax[2].plot(np.arange(len(hist)) * 0.05, np.array(hist) / h0, lw=2)
ax[2].axhline(1, ls=":", c="0.6"); ax[2].set(xlabel="time", ylabel="bed height / initial", title="expansion")
plt.tight_layout()
print(f"bed height {h0:.2f} -> {hist[-1]:.2f}  (x{hist[-1]/h0:.2f})  min voidage {float(np.asarray(cpl.last_eps).min()):.2f}")

The bed expands by half its height and the minimum void fraction climbs off the packed value — the
grains are suspended, not resting. Sweeping the gas velocity traces the classic **fluidization curve**:
flat while the bed is fixed, then rising once `U` clears `U_mf`.

In [ ]:
#| label: fig-curve
#| fig-cap: "Fluidization curve: the bed height is flat while the gas only percolates, then climbs once the drag carries the grain weight (U > U_mf)."
def final_height(U, steps=80):
    s = make_flow(U); d = make_dem(pos0)
    c = CfdDem(s, d, fluid_dt=0.05, mu=mu_g, rho=rho_g, radius=rp, drag="gidaspow",
               dem_substeps=20, eps_min=0.3, periodic=(False, False, False), move_particles=True)
    for _ in range(steps): c.step()
    return bed_top(c)

Us = [0.0, 0.05, 0.10, 0.18, 0.30]
Hs = [final_height(U) for U in Us]
plt.figure(figsize=(4.6, 3.2))
plt.plot(Us, Hs, "o-", lw=2); plt.axhline(Hs[0], ls=":", c="0.6", label="fixed-bed height")
plt.xlabel("superficial gas velocity U"); plt.ylabel("bed height (95th pct)")
plt.title("fluidization curve"); plt.legend(); plt.tight_layout()
print("bed height vs U:", {U: round(h, 2) for U, h in zip(Us, Hs)})

## Is the granular solver fast? Scaling to a million grains

The coupling is only worth it if the DEM keeps up at real bed sizes. `peclet.dem`'s broadphase is
[**ArborX**](https://github.com/arborx/ArborX) — Sandia's performance-portable **bounding-volume
hierarchy** — so contact search is `O(N log N)` and runs on the same device (CUDA / HIP / OpenMP) as
the rest of the step. There is no CPU fallback and no per-frame host copy of the neighbour list.

We time a full XPBD step (broadphase + narrowphase + constraint solve) on a loose pack of unit-radius
grains, scaling `N` by decades. This runs on whatever backend you have — the numbers below are on the
render machine; the commentary quotes a GPU.

In [ ]:
#| label: fig-scaling
#| fig-cap: "DEM throughput vs grain count on this backend. ArborX keeps the per-grain cost roughly flat; on a GPU it *falls* with N (the device saturating) — a million grains at ~10 ms/step, ~100 M grains/s."
def dem_throughput(N, steps=5):
    n = int(round(N ** (1 / 3))) + 1; sp = 2.2
    xs = np.arange(n) * sp; X, Y, Z = np.meshgrid(xs, xs, xs, indexing="ij")
    p = np.stack([X.ravel(), Y.ravel(), Z.ravel()], 1)[:N].astype(np.float32); L = n * sp
    s = dem.Simulation(int(N * 1.1) + 64); s.initialize(shape_type=1, radius=1.0)   # radius-1 grains
    s.set_domain((0, 0, 0), (L, L, 2 * L)); s.enable_periodicity(False, False, False)
    s.set_gravity(0, 0, -9.8); s.set_dt(1e-3); s.set_material_params(0.3, 0.0, 0.3)
    s.set_positions(np.c_[p, np.ones(N, np.float32)]); s.set_velocities(np.zeros((N, 3), np.float32))
    s.step(1e-3)                                   # warm up (first ArborX build)
    t = time.time()
    for _ in range(steps): s.step(1e-3)
    return (time.time() - t) / steps

Ns = [10_000, 100_000, 1_000_000]
ms = [dem_throughput(N) * 1e3 for N in Ns]
for N, t in zip(Ns, ms):
    print(f"  N = {N:>9,}:  {t:8.1f} ms/step   {N / (t/1e3) / 1e6:6.1f} M grains/s")

fig, ax = plt.subplots(1, 2, figsize=(8, 3.2))
ax[0].loglog(Ns, ms, "o-", lw=2); ax[0].set(xlabel="grains N", ylabel="ms / step", title="step time")
ax[0].grid(True, which="both", alpha=.3)
ax[1].semilogx(Ns, [N / (t/1e3) / 1e6 for N, t in zip(Ns, ms)], "o-", lw=2, color="C1")
ax[1].set(xlabel="grains N", ylabel="M grains / s", title="throughput"); ax[1].grid(True, alpha=.3)
plt.tight_layout()

A **million grains** steps in ~0.6 s on four CPU cores here, and ~**10 ms on an RTX 5080** (≈ 100 M
grains/s) — the throughput *rises* with `N` on the GPU as the device fills up, the signature of a
broadphase that parallelizes cleanly. The BVH is emphatically suited to this; the granular solver is
**not** the bottleneck.

The *coupled* step stays **entirely on-device** — `CfdDem` deposits, gathers, and writes the drag
straight into dem's external-force buffer through zero-copy device views, with no per-step host copy —
so the coupling's own overhead is a few percent (~0.1 s at 100k grains). What the coupled step actually
pays for is the two solvers themselves: the **incompressible-flow pressure solve** on the fluid grid,
and the DEM (which costs more in a *dense* bed than the loose scaling pack above — every grain is in
contact). The granular solver has headroom to spare; a bigger bed is limited by the CFD, not the BVH.

::: {.callout-note}
## A million grains, packed
The scaling run above pours a million loose grains; letting them settle produces a dense pack in a few
hundred steps (~a minute on the GPU). Point a `CfdDem` at that pack, widen the vessel to match, and the
whole fluidized bed above runs at bed-scale — same code, more grains.
:::

## Adapt this yourself

- **Change the drag law.** `drag="ergun"` (fixed-bed pressure drop), `"wen_yu"` (dilute), `"di_felice"`,
  `"schiller_naumann"`, `"stokes"` — or `"gidaspow"` here. Each is a device-inline correlation in
  `coupling/src/drag.hpp`; add your own by extending the `DragKind` switch.
- **Change the vessel.** `capped_cylinder_wall` / `cylinder_flow_sdf` are ordinary SDFs — swap the
  cylinder for a cone (spouted bed), add a draft tube, or make the distributor a perforated plate.
- **Go multi-rank.** Build the `peclet-*-mpi` modules, pass `flow.init_mpi` / `dem.init_mpi` the same
  decomposition, and `CfdDem` runs the distributed deposit-fold + gather + DEM step. This bounded,
  driven bed is well-posed, so it reproduces the single-rank run across ranks.
- **Bigger bed.** Raise `NX, NY, NZ, R` and the grain count follows; the DEM has the headroom (above).

## Reproduce this

```bash
# CPU (OpenMP) — flow + dem + the coupling module, built from source against a Kokkos prefix:
tools/bootstrap_deps.sh host-openmp                      # one-time Kokkos/ArborX prefix
for m in flow dem coupling; do
  cmake -S $m -B $m/build -DCMAKE_PREFIX_PATH="$PWD/extern/install/host-openmp"; cmake --build $m/build -j
done
PECLET_LOCAL_BUILD="$PWD/flow/build:$PWD/dem/build:$PWD/coupling/build" \
  quarto render examples/fluidized-bed/index.qmd
# GPU: point CMAKE_PREFIX_PATH at extern/install/nvidia-cuda instead (nvcc on PATH).
```